In [1]:
%pip install pypdf sentence-transformers faiss-cpu numpy pandas scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import re
import numpy as np
import faiss
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("Setup complete! Libraries imported successfully.")

Setup complete! Libraries imported successfully.


In [3]:
# Load your PDF file 
# (Make sure your PDF is in the same folder as this notebook file!)
pdf_filename = "UG-Prospectus-2024.pdf"  # <-- CHANGE THIS to your actual PDF file name

print(f"Reading text from {pdf_filename}...")
reader = PdfReader(pdf_filename)

raw_text = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        raw_text += text + "\n"

# Clean up messy spaces and newlines into uniform single spaces
clean_text = re.sub(r'\s+', ' ', raw_text).strip()
words = clean_text.split()

print(f"Text extraction done!")
print(f"Total Words extracted: {len(words)}")

Reading text from UG-Prospectus-2024.pdf...
Text extraction done!
Total Words extracted: 94585


<center><h1>FIXED SIZE CHUNKING</h1></center>

In [4]:
fixed_chunks = []
chunk_size = 200  # Number of words per chunk

for i in range(0, len(words), chunk_size):
    # Take a slice of 200 words and join them into a paragraph string
    chunk = " ".join(words[i : i + chunk_size])
    fixed_chunks.append(chunk)

print(f"Strategy 1 Complete: Generated {len(fixed_chunks)} chunks.")
print(f"Sample Chunk 1:\n{fixed_chunks[0][:150]}...")

Strategy 1 Complete: Generated 473 chunks.
Sample Chunk 1:
VISION The GIK Institute seeks to play an inspiring role in imparting high-quality education and research in the fields of engineering, sciences, emer...


<center><h1 style="color:yellow">FIXED SIZE CHUNKING + OVERLAP</h1></center>

In [5]:
overlap_chunks = []
chunk_size = 200
overlap = 50
step = chunk_size - overlap  # Advances 150 words forward each time

for i in range(0, len(words), step):
    chunk = " ".join(words[i : i + chunk_size])
    overlap_chunks.append(chunk)
    
    # Break out if we've completely reached the end of the text
    if i + chunk_size >= len(words):
        break

print(f"Strategy 2 Complete: Generated {len(overlap_chunks)} chunks.")
print(f"Sample Chunk 1:\n{overlap_chunks[0][:150]}...")

Strategy 2 Complete: Generated 631 chunks.
Sample Chunk 1:
VISION The GIK Institute seeks to play an inspiring role in imparting high-quality education and research in the fields of engineering, sciences, emer...


<center><h1 style="color:yellow">SENTENCE LEVEL CHUNKING</h1></center>

In [6]:
# A simple regular expression that splits at structural punctuation (. ! ?) followed by a space
sentence_chunks = re.split(r'(?<=[.!?])\s+', clean_text)

# Remove any empty items
sentence_chunks = [s.strip() for s in sentence_chunks if s.strip()]

print(f"Strategy 3 Complete: Generated {len(sentence_chunks)} sentence chunks.")
print(f"Sample Sentence 1:\n{sentence_chunks[0]}")

Strategy 3 Complete: Generated 3731 sentence chunks.
Sample Sentence 1:
VISION The GIK Institute seeks to play an inspiring role in imparting high-quality education and research in the fields of engineering, sciences, emerging technologies, and other disciplines.


<center><h1 style="color:yellow">PARAGRAPH LEVEL / SEMANTIC CHUNKING</h1></center>

In [7]:
# ==========================================
# STRATEGY 4: SEMANTIC / PARAGRAPH HEURISTIC
# ==========================================

# Step 1: Split the raw text on double newlines or paragraph breaks
# We use re.split to catch situations where there are multiple newlines between sections
paragraph_chunks_raw = re.split(r'\n\s*\n', raw_text)

semantic_chunks = []
current_chunk = []
current_word_count = 0
target_word_limit = 300  # Merge small consecutive paragraphs into a meaningful topic block

for paragraph in paragraph_chunks_raw:
    clean_paragraph = re.sub(r'\s+', ' ', paragraph).strip()
    
    # Skip empty lines
    if not clean_paragraph:
        continue
        
    paragraph_words = clean_paragraph.split()
    
    # Heuristic Rule: If adding this paragraph exceeds our topic size limit, 
    # finalize the current chunk and start a fresh topic block.
    if current_word_count + len(paragraph_words) > target_word_limit and current_chunk:
        semantic_chunks.append(" ".join(current_chunk))
        current_chunk = [clean_paragraph]
        current_word_count = len(paragraph_words)
    else:
        # Otherwise, keep grouping these paragraphs together as one continuous topic
        current_chunk.append(clean_paragraph)
        current_word_count += len(paragraph_words)

# Catch the final leftover topic group
if current_chunk:
    semantic_chunks.append(" ".join(current_chunk))

print(f"🧠 Strategy 4 Complete: Generated {len(semantic_chunks)} paragraph-based topic chunks.")

🧠 Strategy 4 Complete: Generated 142 paragraph-based topic chunks.


In [8]:
import faiss

# Initialize our top-performing model
print("🤖 Loading all-mpnet-base-v2...")
model = SentenceTransformer('all-mpnet-base-v2')

# 1. Generate embeddings for all 4 strategies
print("⏳ Generating vectors (this might take a few moments)...")
vectors_fixed    = model.encode(fixed_chunks, show_progress_bar=True)
vectors_overlap  = model.encode(overlap_chunks, show_progress_bar=True)
vectors_sentence = model.encode(sentence_chunks, show_progress_bar=True)
vectors_semantic = model.encode(semantic_chunks, show_progress_bar=True)

# 2. Get the vector dimension size (mpnet is 768)
dimension = vectors_fixed.shape[1]

# 3. Create and populate the 4 separate FAISS Indexes
# We use IndexFlatIP (Inner Product) on normalized vectors because it equals Cosine Similarity
def build_faiss_index(embeddings):
    # Create the raw index
    index = faiss.IndexFlatIP(dimension)
    # Normalize vectors to unit length so inner product acts as cosine similarity
    faiss.normalize_L2(embeddings)
    # Add vectors to our index
    index.add(embeddings)
    return index

index_fixed    = build_faiss_index(vectors_fixed.astype('float32'))
index_overlap  = build_faiss_index(vectors_overlap.astype('float32'))
index_sentence = build_faiss_index(vectors_sentence.astype('float32'))
index_semantic = build_faiss_index(vectors_semantic.astype('float32'))

print("✅ All 4 FAISS Vector Indexes built successfully!")

🤖 Loading all-mpnet-base-v2...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

⏳ Generating vectors (this might take a few moments)...


Batches:   0%|          | 0/15 [00:00<?, ?it/s]

Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Batches:   0%|          | 0/117 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

✅ All 4 FAISS Vector Indexes built successfully!


In [9]:
# List of your 5 targeted Docker queries
test_queries = [
    "What are the eligibility criteria and admission requirements for undergraduate programs at GIKI, including SAT/entry test requirements and minimum academic qualifications?",
    "What undergraduate degree programs does GIKI offer, and what is the credit hour breakdown between general education, core, and elective courses?",
    "What are the tuition fees, hostel charges, and available scholarship or financial assistance options for students at GIKI?",
    "Which faculties and departments exist at GIKI, who are the current Deans and Heads of Departments, and what research areas do they specialize in?",
    "What facilities, extracurricular activities, student societies, and residential accommodations are available for students at GIK Institute?"
]

def search_index(query_text, faiss_index, raw_chunks_list):
    """Encodes a query and returns the best matching chunk and its similarity score."""
    # Encode query and normalize it
    query_vector = model.encode([query_text]).astype('float32')
    faiss.normalize_L2(query_vector)
    
    # Search for the Top-1 closest match (k=1)
    scores, indices = faiss_index.search(query_vector, k=1)
    
    best_idx = indices[0][0]
    best_score = scores[0][0]
    
    # Return the text chunk and its similarity score
    return raw_chunks_list[best_idx], best_score

In [11]:
# Dictionary to make looping clean
strategies = {
    "Fixed-Size (200 words)": (index_fixed, fixed_chunks),
    "Fixed-Size + Overlap": (index_overlap, overlap_chunks),
    "Sentence-Level": (index_sentence, sentence_chunks),
    "Semantic/Paragraph": (index_semantic, semantic_chunks)
}

print(" Running Retrieval Evaluation...\n")

for q_idx, query in enumerate(test_queries, 1):
    print("=" * 90)
    print(f" QUERY {q_idx}: '{query}'")
    print("=" * 90)
    
    for name, (index, chunks) in strategies.items():
        retrieved_text, score = search_index(query, index, chunks)
        
        # Calculate word count of the retrieved chunk
        chunk_words = len(retrieved_text.split())
        
        print(f"\n Strategy: {name} | Score: {score:.4f} | Size: {chunk_words} words")
        print(f" Retrieved Chunk:\n   \"{retrieved_text[:300]}...\"")
        print("-" * 50)

 Running Retrieval Evaluation...

 QUERY 1: 'What are the eligibility criteria and admission requirements for undergraduate programs at GIKI, including SAT/entry test requirements and minimum academic qualifications?'

 Strategy: Fixed-Size (200 words) | Score: 0.6278 | Size: 200 words
 Retrieved Chunk:
   "mission and goal of achieving sustainable targets and continuing to progress in this area, and its achievement in the GreenMetric Rankings reaffirms its commitment to implementing environmentally friendly policies and “greening” the campus to support sustainability. Accreditation of Programs GIK Ins..."
--------------------------------------------------

 Strategy: Fixed-Size + Overlap | Score: 0.7508 | Size: 200 words
 Retrieved Chunk:
   "of 4.0, may apply to this Institute for admission with advanced standing. However, the student at the GIK Institute, to qualify for a bachelor degree, must earn a minimum of 70 credits including 6 credit hours of senior year design project. An ap